In [0]:
# Paths — these stay the same across all notebooks
RAW_PATH    = "/Volumes/workspace/default/cyclistic_data/raw"
BRONZE_PATH = "/Volumes/workspace/default/cyclistic_data/bronze"
SILVER_PATH = "/Volumes/workspace/default/cyclistic_data/silver"
GOLD_PATH   = "/Volumes/workspace/default/cyclistic_data/gold"

# Name of our Bronze Delta table
BRONZE_TABLE = "workspace.default.bronze_rides"

print("Paths configured:")
print(f"  Raw:    {RAW_PATH}")
print(f"  Bronze: {BRONZE_PATH}")
print(f"  Silver: {SILVER_PATH}")
print(f"  Gold:   {GOLD_PATH}")
print(f"\nBronze table: {BRONZE_TABLE}")

In [0]:
# Read all 12 CSV files at once into a Spark DataFrame
# inferSchema=True means Spark will figure out column types automatically
# header=True means the first row of each CSV is column names

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{RAW_PATH}/*.csv")
)

# Show how many rows and columns we have
print(f"Total rows:    {df_raw.count():,}")
print(f"Total columns: {len(df_raw.columns)}")
print(f"\nColumn names:")
for col in df_raw.columns:
    print(f"  - {col}")

In [0]:
# Show the first 5 rows so we can see what the data looks like
display(df_raw.limit(5))

In [0]:
# Show what data type Spark assigned to each column
print("Column types:\n")
for col_name, col_type in df_raw.dtypes:
    print(f"  {col_name:<25} {col_type}")

In [0]:
# Save the raw data as a Delta table
# This is our Bronze layer — raw data, nothing changed, just stored properly

(df_raw
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze table saved successfully!")
print(f"Table name: {BRONZE_TABLE}")

In [0]:
# Read back from the Delta table to confirm it saved correctly
df_check = spark.table(BRONZE_TABLE)

row_count = df_check.count()
col_count = len(df_check.columns)

print(f"Bronze table verified!")
print(f"  Rows:    {row_count:,}")
print(f"  Columns: {col_count}")
print(f"\nSample data:")
display(df_check.limit(3))